In [1]:
!pip install --upgrade -q kagglehub

import kagglehub
import os
import json
import pandas as pd

print("Fetching the complete Spider archive via kagglehub")
archive_path = kagglehub.dataset_download("jeromeblanchet/yale-universitys-spider-10-nlp-dataset")

print(f"Archive downloaded successfully and cached at: {archive_path}")

db_source_dir = os.path.join(archive_path, "spider", "database")
if not os.path.exists(db_source_dir):
    db_source_dir = os.path.join(archive_path, "database")

dev_json_path = os.path.join(archive_path, "spider", "dev.json")
if not os.path.exists(dev_json_path):
    dev_json_path = os.path.join(archive_path, "dev.json")

print(f"Verified Database Path: {db_source_dir}")
print(f"Verified dev.json Path: {dev_json_path}")

Fetching the complete Spider archive via kagglehub


100%|██████████| 96.0M/96.0M [00:06<00:00, 15.1MB/s]

Extracting files...


Archive downloaded successfully and cached at: /root/.cache/kagglehub/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/versions/1
Verified Database Path: /root/.cache/kagglehub/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/versions/1/spider/database
Verified dev.json Path: /root/.cache/kagglehub/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/versions/1/spider/dev.json


In [2]:
# Check databases
available_dbs = os.listdir(db_source_dir) if os.path.exists(db_source_dir) else []
print(f"Total Database Schemas Discovered: {len(available_dbs)}")

# Check evaluation targets
if os.path.exists(dev_json_path):
    with open(dev_json_path, 'r') as f:
        spider_dev_data = json.load(f)
    print(f"Original spider dev rows verified: {len(spider_dev_data)}")
else:
    print("Error tracking down your dev.json location.")

Total Database Schemas Discovered: 166
Original spider dev rows verified: 1034


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading local Qwen-1.5B-Coder model onto VRAM")
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda"
)
print("Local Transformer model is successfully locked into GPU memory!")

Loading local Qwen-1.5B-Coder model onto VRAM


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Local Transformer model is successfully locked into GPU memory!


In [4]:
with open("/content/hinglish_textosql_final.json", "r", encoding="utf-8") as f:
    sample_data = json.load(f)
print("Keys available in your dataset rows:", sample_data[0].keys())
print("Sample row content:", sample_data[0])

Keys available in your dataset rows: dict_keys(['db_id', 'english', 'hinglish_light', 'hinglish_natural', 'hinglish_hindi_heavy', 'sql'])
Sample row content: {'db_id': 'film_rank', 'english': 'Return the low and high estimates for all film markets.', 'hinglish_light': 'Return the chota aur bade teshimate for all film market', 'hinglish_natural': 'Return chota aur bade teshimate for all film market', 'hinglish_hindi_heavy': 'Return the chota aur bade teshimate for all film market', 'sql': 'SELECT Low_Estimate ,  High_Estimate FROM film_market_estimation'}


In [6]:
import json

with open("hinglish_textosql_final.json", "r", encoding="utf-8") as f:
    eval_dataset = json.load(f)

In [9]:
import os
import sqlite3
import re
import torch
import pandas as pd
from tqdm import tqdm

db_source_dir = "/root/.cache/kagglehub/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/versions/1/spider/database"

def get_db_schema(db_path):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
        tables = [row[0] for row in cursor.fetchall() if row[0] is not None]
        conn.close()
        return "\n".join(tables)
    except Exception:
        return ""

def execute_sql(db_path, query):
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        cursor.execute(query)
        results = cursor.fetchall()
        conn.close()
        return set(results), "SUCCESS"
    except Exception as e:
        return None, str(e)

def generate_local_sql(system_prompt, user_query):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Output only raw SQL code for: {user_query}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=150,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.1
        )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

results_log = []
missing_db_counter = 0

print("Starting Optimized Dataset GPU Evaluation Loop...")
for row in tqdm(eval_dataset, desc="Evaluating Linguistic Tiers"):
    db_id = row['db_id']
    golden_sql = row['sql']

    db_path = os.path.join(db_source_dir, db_id, f"{db_id}.sqlite")
    if not os.path.exists(db_path):
        missing_db_counter += 1
        continue

    db_schema = get_db_schema(db_path)

    system_instruction = (
        f"You are a strict Text-to-SQL system running on SQLite.\n"
        f"Database Schema:\n{db_schema}\n\n"
        f"Translate the text to a valid SQL query. Do not explain, do not write introduction text, do not use markdown blocks. Output only the pure raw SQL query text string."
    )

    for tier in ['hinglish_light', 'hinglish_natural', 'hinglish_hindi_heavy']:
        generated_raw = generate_local_sql(system_instruction, row[tier])

        sql_lines = [line.strip() for line in generated_raw.split("\n") if line.strip()]
        clean_sql = ""
        for line in sql_lines:
            if line.upper().startswith("SELECT"):
                clean_sql = line
                break
        if not clean_sql and sql_lines:
            clean_sql = sql_lines[-1]

        clean_sql = clean_sql.replace("```sql", "").replace("```", "").strip()
        clean_sql = clean_sql.split(";")[0].strip()

        golden_res, golden_status = execute_sql(db_path, golden_sql)
        gen_res, gen_status = execute_sql(db_path, clean_sql)

        is_correct = (golden_res == gen_res) if golden_res is not None and gen_res is not None else False

        results_log.append({
            "db_id": db_id,
            "tier": tier,
            "hinglish_input": row[tier],
            "golden_sql": golden_sql,
            "generated_sql": clean_sql,
            "execution_match": is_correct,
            "status": gen_status if not is_correct else "CLEAN"
        })

eval_df = pd.DataFrame(results_log)
if not eval_df.empty:
    summary = eval_df.groupby('tier')['execution_match'].mean() * 100
    print("\n" + "="*40)
    print("--- FINAL LOCAL TRANSFORMER SCHEMATIC ACCURACY REPORT ---")
    print("="*40)
    print(summary.to_string(formatters={lambda x: f"{x:.6f}%"}))
    print("="*40)
    if missing_db_counter > 0:
        print(f"Skipped {missing_db_counter} evaluations due to missing database paths.")
else:
    print("Execution metrics compilation failed.")

Starting Optimized Dataset GPU Evaluation Loop...


Evaluating Linguistic Tiers: 100%|██████████| 1465/1465 [1:23:06<00:00,  3.40s/it]


--- FINAL LOCAL TRANSFORMER SCHEMATIC ACCURACY REPORT ---


TypeError: Series.to_string() got an unexpected keyword argument 'formatters'

In [13]:
# Convert results_log into a DataFrame if it hasn't been re-assigned
eval_df = pd.DataFrame(results_log)

if not eval_df.empty:
    summary = eval_df.groupby('tier')['execution_match'].mean() * 100
    print("zero-shot schematic accuracy report for qwen2.5-coder-1.5b-instruct")
    # Using float_format with an anonymous lambda function for Series strings
    print(summary.to_string(float_format=lambda x: f"{x:.6f}%"))
    if missing_db_counter > 0:
        print(f"Skipped {missing_db_counter} evaluations due to missing database paths.")
else:
    print("Execution metrics compilation failed.")

zero-shot schematic accuracy report for qwen2.5-coder-1.5b-instruct
tier
hinglish_hindi_heavy   20.136519%
hinglish_light         28.668942%
hinglish_natural       20.273038%
